# データベース演習 第12回

### 🔍 利用方法

- このノートブックは**閲覧専用**です。  
  自分のGoogleドライブにコピーを作成してから編集してください。

  1. メニューの「**ファイル → ドライブにコピーを保存**」を選ぶ  
  2. コピーしたノートブックの**ファイル名（左上の"xxxx"）を学籍番号に変更**してください（提出時の識別のため）

---

### 🔧 提出手順

1. Google Colab の右上にある「**共有**」ボタンをクリック  
2. 「**一般的なアクセス**」の設定を「**リンクを知っている全員**」に変更  
3. アクセス権を「**閲覧者**」に設定（⚠️「編集者」にしないこと！）  
4. 表示されるURLをコピー  
5. WebClassの提出フォームに、その**URLを貼り付けて提出**

---

### 💡 補足・注意点

- **「編集者」ではなく「閲覧者」**に設定してください。  
  → 教員が間違ってファイルを上書きしてしまうのを防ぐためです。
- **提出したリンクを自分でも一度開いてみて、ちゃんと共有されているかを確認**してください。

### データベースをダウンロード

* 全データが含まれています
* Colabのランタイムが切り替わると，その度にダウンロードする必要があります
* その度にダウンロードするのが面倒な人は，自分のGoogleドライブに保存し，そのファイルを参照する方法もあります（生成AIなどで調べてみてください）．

In [ ]:
# Google DriveからSQLiteファイルを取得
import gdown

# ファイルIDを指定
file_id = '1nuCOXMd3H0rL-82D460SnbpM7YjLNBeq'
url = f'https://drive.google.com/uc?id={file_id}'

# 保存ファイル名を指定
output = 'weblog.sqlite3'
gdown.download(url, output, quiet=False)

### JupySQLのインストールと有効化

In [ ]:
# Colabではjupysqlのインストールが必要
import sys
if 'google.colab' in sys.modules:
    %pip install -q jupysql

In [ ]:
# jupysqlの拡張機能を有効化`
%load_ext sql

In [ ]:
# 結果表示数は以下の数値を変えれば変更できる
%config SqlMagic.displaylimit = 10 # デフォルトは10行

### Webログデータベース

In [ ]:
# Webログデータベースに接続する
%sql sqlite:///weblog.sqlite3

### 含まれるテーブルの確認

In [ ]:
%%sql
SELECT * FROM sqlite_master;


### 例題1

(1) ordersテーブルには，顧客のIDを表すcustomer_idカラムがある．また，customersテーブルにもcustomer_idカラムはあるが，さらに，顧客が住む都道府県を表すcustomer_locationカラムがある．

そこで，サブクエリを利用し，新潟県に住んでいる顧客の注文をordersテーブルから求めるSQL文を作成せよ．ただし，結果の行数は10に限定せよ．

In [ ]:
%%sql
SELECT * FROM orders WHERE customer_id IN (
	SELECT customer_id FROM customers WHERE customer_location='新潟県'
) LIMIT 10;

(2) ordersテーブルを用いて，顧客のID（customer_id），顧客毎の注文金額の最大値，全体での注文金額の最大値の組を表示するSQL文を作成せよ．ただし，結果の行数は10に限定せよ．

In [ ]:
%%sql
SELECT customer_id, MAX(order_amount), (SELECT MAX(order_amount) FROM orders)
FROM orders GROUP BY customer_id LIMIT 10;

## 例題2

(3) itemsテーブルは，商品を管理しているテーブルで，商品のID（item_id），ショップのID（shop_id），商品の名前（item_name），商品の価格（item_price）をカラムとして持っている．ウィンドウ関数AVGを用いて，商品のID，ショップのID，商品の価格，ショップ内における商品価格の平均，の組を求めるSQL文を作成せよ．検索結果は最初の10件に限定せよ．

In [ ]:
%%sql
SELECT item_id, shop_id, item_price,
	AVG(item_price) OVER (
		PARTITION BY shop_id
	)
FROM items LIMIT 10;

## 演習：課題6-1

(1) ordersテーブルには，顧客のIDを表すcustomer_idカラムがある．また，customersテーブルにもcustomer_idカラムはあるが，さらに，顧客の年齢を表すcustomer_ageカラムがある．

そこで，サブクエリを利用し，年齢が30歳より低い（30歳は含まない）顧客の注文をordersテーブルから求めるSQL文を作成せよ．ただし，結果の行数は10に限定せよ．


In [ ]:
%%sql


(2) itemsテーブルは，商品に関する情報のテーブルである，商品を売っているショップのID（shop_id），商品の価格（item_price）というカラムを持っている．

そこで，ショップのID，ショップ毎の商品の価格の平均値，全体での商品の価格の平均値の組を表示するSQL文を作成せよ．ただし，結果の行数は10に限定せよ


In [ ]:
%%sql


## 演習 課題6-2

(3) customersテーブルは，顧客を管理しているテーブルで，顧客のID（customer_id），顧客の年齢（customer_age），顧客の住む場所（都道府県）（customer_location）等をカラムとして持っている．ウィンドウ関数AVGを用いて，顧客のID，顧客の住む場所，顧客の年齢，同じ場所（都道府県）に住む顧客の年齢の平均，の組を求めるSQL文を作成せよ．検索結果は最初の10件に限定せよ．以下に結果例（一部）を示す．

In [ ]:
%%sql
